# Birbank public product scraper
Collect cards, loans, deposits, insurance, leasing, transfers and public tariff documents from **birbank.az**. Run the cells in order.

Each run creates a new folder under `birbank_data` containing `products.csv`, full JSON data, original HTML and tariff documents, and `manifest.json` with any missing pages. Fees and rates retain their published conditions. External application sites and interactive calculators are not crawled.


In [ ]:
%pip install -r requirements.txt


In [ ]:
import importlib
import web
importlib.reload(web)  # Reload if an older scraper was imported in this kernel.

MAX_PAGES = 100
DELAY_SECONDS = 1.0
DOWNLOAD_DOCUMENTS = True


In [ ]:
result = web.scrape_birbank(
    output="birbank_data", max_pages=MAX_PAGES,
    delay=DELAY_SECONDS, documents=DOWNLOAD_DOCUMENTS,
)
print("Output:", result["folder"].resolve())
print("Coverage:", result["manifest"]["coverage"])
print("Errors:", result["manifest"]["errors"])


In [ ]:
[{"name": p["name"], "category": p["category"],
  "price_rate_mentions": p["price_rate_mentions"], "source": p["source_url"]}
 for p in result["products"][:10]]


# Birmarket real products for Superchat
Run this section to collect real product details in the structure of `SuperChat_Mock_Data/products.json`. Original mock data and the Birbank scraper remain available.

Output: `SuperChat_Real_Data/birmarket/products.json`, `products.csv`, and a coverage manifest. Image galleries use real image URLs. Set `DOWNLOAD_IMAGES = True` to download every gallery image as well. Exact stock and unavailable translations are `None`.


In [ ]:
%pip install -r requirements.txt


In [ ]:
import importlib
import birmarket_scraper
importlib.reload(birmarket_scraper)

PER_CATEGORY = 20  # 0 = all products within PAGES.
PAGES = 2
DOWNLOAD_IMAGES = True
FETCH_RUSSIAN = True


In [ ]:
birmarket_result = birmarket_scraper.scrape_catalog(
    per_category=PER_CATEGORY, pages=PAGES, delay=1.0,
    output="SuperChat_Real_Data", russian=FETCH_RUSSIAN,
    download_images=DOWNLOAD_IMAGES,
)
print(birmarket_result["manifest"])


In [ ]:
products = birmarket_result["products"]
products[0] if products else "No products collected"


In [ ]:
from pathlib import Path
from IPython.display import Image, display

# Prefer a downloaded picture; otherwise use its source URL.
if products:
    if products[0]["image_files"]:
        image_path = Path("SuperChat_Real_Data") / products[0]["image_files"][0]
        display(Image(filename=str(image_path), width=300))
    elif products[0]["image_url"]:
        display(Image(url=products[0]["image_url"], width=300))


# Aviasales real flight fares for Superchat
Collect the available dated fares on the nine routes in `aviasales_routes.json`. These are **cached advertised fares**, not live seat availability. Actual published dates are retained; unknown flight numbers, cabin, baggage weight and seat counts stay `None`. One-way and round-trip prices are exported separately.


In [ ]:
import importlib
import aviasales_scraper
importlib.reload(aviasales_scraper)

FLIGHT_FROM_DATE = None  # Optional YYYY-MM-DD filter on published departure dates
FLIGHT_TO_DATE = None


In [ ]:
flight_result = aviasales_scraper.scrape_flights(
    config_path="aviasales_routes.json", output="SuperChat_Real_Data",
    delay=1.0, from_date=FLIGHT_FROM_DATE, to_date=FLIGHT_TO_DATE,
)
print("One-way fares:", len(flight_result["flights"]))
print("Round-trip fares:", len(flight_result["round_trips"]))
print("Coverage:", flight_result["manifest"]["coverage"])
print("Errors:", flight_result["manifest"]["errors"])


In [ ]:
[{"airline": f["airline"], "from": f["origin"]["city"],
  "to": f["destination"]["city"], "departure": f["departure_local"],
  "arrival": f["arrival_local"], "price_azn": f["price_azn"],
  "stops": f["stops"], "nearby_airport": f["nearby_airport_search"],
  "search_url": f["search_url"]}
 for f in flight_result["flights"]]


Use `SuperChat_Real_Data/aviasales/flights.json` for one-way flight cards. City/search codes do not confirm exact airports. A Dubai route offer that searches Sharjah is labeled Sharjah, with the original route retained in `source_route`. Recheck prices via the search link before booking. Full source HTML and collection details are saved under `aviasales/runs/`.


# CineMastercard cinema dataset
Collect real movie details, cinemas and published dated ticket prices from Cinemastercard. Posters and tariff images are downloaded; seats and ratings remain unknown. See `SuperChat_Real_Data/README.md` and `DATA_DICTIONARY.md` for dataset notes.


In [ ]:
import importlib
import cinema_scraper
importlib.reload(cinema_scraper)

cinema_result = cinema_scraper.scrape_cinema(
    output="SuperChat_Real_Data", delay=0.75, download_images=True,
)
cinema_result["manifest"]


In [ ]:
movie_lookup = {m["id"]: m for m in cinema_result["movies"]}
cinema_lookup = {c["id"]: c for c in cinema_result["cinemas"]}
[{"movie": movie_lookup[s["movie_id"]]["title"],
  "cinema": cinema_lookup[s["cinema_id"]]["name"],
  "date": s["date"], "time": s["time"], "hall": s["hall"],
  "price_azn": s["price_azn"], "price_conflict": s["price_conflict"],
  "languages": s["languages"], "formats": s["formats"]}
 for s in cinema_result["showtimes"][:20]]


In [ ]:
from validate_cinema import validate_dataset
validate_dataset("SuperChat_Real_Data")
